# 2. Ghép cache thành dataset
CPU, Internet On. Attach RLDD Full + Face Landmarker + output của đủ 4 nhóm trích xuất. Không có MediaPipe extraction mới ở bước này.

In [ ]:
import sys, os, json, subprocess, urllib.request, zipfile
from pathlib import Path
INPUT, WORK = Path('/kaggle/input'), Path('/kaggle/working')
REV = '8aeacc1'
archive = WORK/'code20.zip'
urllib.request.urlretrieve(f'https://github.com/toafn11/driver-monitoring-system/archive/{REV}.zip', archive)
destination = WORK/f'code_{REV}'
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        if not (destination/name).resolve().is_relative_to(destination.resolve()):
            raise ValueError('Unsafe archive path')
    z.extractall(destination)
CODE = next(destination.rglob('prepare20.py')).parents[1]
os.chdir(CODE)
sys.path.insert(0, str(CODE))
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(CODE/'requirements-kaggle20.txt')],check=True)
print('Code revision:', REV)


In [ ]:
from training.prepare20 import rldd_manifest
folds = sorted(INPUT.rglob('Fold1_part1'))
if not folds: raise FileNotFoundError('Attach thuannn18022005/uta-rldd-full')
ROOT = folds[0].parent
models = sorted(INPUT.rglob('face_landmarker.task'))
if len(models) != 1: raise ValueError('Attach exactly one Face Landmarker model')
MODEL = models[0]
MANIFEST = WORK/'manifest20.csv'
rldd_manifest(ROOT, MANIFEST)
SAMPLE_FPS = 10  # Confirm against the audit; identical in all shards and build.
CACHE = WORK/'cache20_v3'
CACHE_SOURCES = sorted(INPUT.rglob('cache20_v3'))
print('Model:', MODEL, 'Prior cache inputs:', CACHE_SOURCES)


In [ ]:
from training.prepare20 import build
markers = [json.loads(p.read_text()) for p in INPUT.rglob('shard_*_complete.json')]
matching = [m for m in markers if m['sample_fps']==SAMPLE_FPS and m['num_shards']==4 and m['revision']==REV]
if {m['shard'] for m in matching} != set(range(4)):
    raise ValueError('Attach completed outputs of all four matching extraction shards')
DATA = WORK/'processed20_v3'
# Four-second sequences, one-second stride. No frame duplication.
build(MANIFEST, MODEL, DATA, CACHE, sample_fps=SAMPLE_FPS,
      seq_len=int(4*SAMPLE_FPS), stride=int(SAMPLE_FPS),
      cache_sources=CACHE_SOURCES, cache_only=True)
print('Ready:', DATA/'metadata.json')
